# ENSEMBLE QUANTUM — K-Fold Cross-Validation + Ensemble
### All 13 Quantum Models | Conjunctiva Image → Hb Estimation

**Workflow (strict — test set never touched during training):**

```
All Data
  │
  ├── Test Set (held-out, fixed) ──────────────────────────────► Final eval only
  │
  └── Train+Val Set
        │
        └── K-Fold Cross-Validation (per model)
              ├── Fold 1: train on K-1, predict OOF on fold 1
              ├── Fold 2: train on K-1, predict OOF on fold 2
              ├── ...
              └── Fold K: → collect OOF predictions for ALL models
                              │
                              ├── Weighted Average Ensemble  (by CV score)
                              ├── Stacking Ensemble          (meta-learner on OOF)
                              └── Best-Single-Model
                                        │
                                        └── Retrain on full Train+Val → Predict Test
```

| # | Model | Type | Publication |
|---|-------|------|-------------|
| 1 | QSVM | SVM | Nature 567 (2019) — IBM |
| 2 | VQC | Gradient | arXiv 1802.06002 (2018) — Google |
| 3 | QCNN | Gradient | Nature Physics 15 (2019) — Harvard |
| 4 | TorchQuantum VQA | Gradient | Nat. Rev. Physics 3 (2021) — MIT |
| 5 | Data Re-Uploading | Gradient | Quantum 4, 226 (2020) |
| 6 | Quantum Transfer Learning | Gradient | Quantum 4, 340 (2020) — Xanadu |
| 7 | Quantum Kernel | SVM | PRX Quantum (2021) — Xanadu |
| 8 | Quantum Random Forest | SVM | Springer QMI (2023) |
| 9 | QBoost | SVM | ACML 2012 — D-Wave |
| 10 | CV-QNN | Gradient | Phys Rev Research 1 (2019) — Xanadu |
| B1 | QSVM Medical | SVM | Bonus |
| B2 | QCNN Phases | Gradient | Bonus |
| B3 | Meta-Learning QNN | Gradient | Bonus |


## Section 1 — CONFIGURATION  (Edit only this cell)

In [ ]:
# ==============================================================
#         ALL VARIABLES — EDIT ONLY THIS BLOCK
# ==============================================================

# ── Task ──────────────────────────────────────────────────────
# "classification"  →  Anemic vs Normal   (binary)
# "regression"      →  predict Hb g/dL   (continuous)
TASK = "regression"

# ── Dataset ───────────────────────────────────────────────────
IMAGE_DIR  = "/kaggle/input/your-dataset/images"
EXCEL_PATH = "/kaggle/input/your-dataset/data.xlsx"
IMAGE_COL  = "Patient_ID"
HB_COL     = "Hemoglobin"

# ── Thresholds & filters ─────────────────────────────────────
HB_THRESH      = 12.0     # g/dL → below = Anemic (for classification)
HB_FILTER_MIN  = 5.0
HB_FILTER_MAX  = 18.0

# ── K-Fold settings ───────────────────────────────────────────
K_FOLDS        = 5        # number of cross-validation folds
TEST_SPLIT     = 0.15     # fraction held out as final test set (never seen in CV)
SEED           = 42

# ── Quantum circuit ───────────────────────────────────────────
N_QUBITS   = 4
N_LAYERS   = 2

# ── Backbone ──────────────────────────────────────────────────
FREEZE_BACKBONE = True

# ── Training (per fold — keep low for CPU quantum simulation) ─
EPOCHS_PER_FOLD = 8       # gradient models: epochs per CV fold
EPOCHS_FINAL    = 15      # gradient models: epochs for final retraining on full train+val
BATCH_SIZE      = 16
LR              = 1e-3
EARLY_STOP      = 3       # patience per fold (0 = disabled)

# ── Regression config ─────────────────────────────────────────
REG_CONFIG = {
    "normalize_hb": True,
    "hb_min": 4.0, "hb_max": 20.0,
    "loss_fn": "huber", "huber_delta": 1.0,
}

# ── Classification config ─────────────────────────────────────
CLS_CONFIG = {
    "use_focal_loss": True, "focal_gamma": 2.0,
    "use_class_weights": True, "threshold": 0.5,
}

# ── Which MODELS to include ───────────────────────────────────
#    Set False (or comment out the line) to skip a model
RUN = {
    "QSVM"                   : True,
    "VQC"                    : True,
    "QCNN"                   : True,
    "TorchQuantum_VQA"       : True,
    "DataReUploading"        : True,
    "QuantumTransferLearning" : True,
    "QuantumKernel"          : True,
    "QuantumRandForest"      : True,
    "QBoost"                 : True,
    "CVQNN"                  : True,
    "QSVM_Medical"           : True,
    "QCNN_Phases"            : True,
    "MetaLearning_QNN"       : True,
}

# ── Which ENSEMBLE METHODS to run ────────────────────────────
#    Set False (or comment out the line) to skip an ensemble method
RUN_ENSEMBLE = {
    "simple_avg"       : True,   # plain mean of all model predictions
    "weighted_avg"     : True,   # weight by CV score (1/MAE or AUC)
    "rank_avg"         : True,   # average of prediction ranks (robust to outliers)
    "stacking_ridge"   : True,   # Ridge/LogReg meta-learner trained on OOF matrix
    "stacking_gbm"     : True,   # GradientBoosting meta-learner on OOF matrix
    "voting"           : True,   # hard majority vote (cls) / median (reg)
    "best_single"      : True,   # single best model by CV score (baseline reference)
}


## Step 0 — Clone Repo  *(run once on Kaggle / Colab)*
> On local: skip this cell if you already have the repo checked out.

In [ ]:
import os, sys, subprocess

REPO_URL  = "https://github.com/junaidmaqbool/QuantumHb.git"
CLONE_DIR = "/kaggle/working/QuantumHb"

if not os.path.isdir(CLONE_DIR):
    print("Cloning QuantumHb ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)
    print(f"✓ Cloned to {CLONE_DIR}")
else:
    print("✓ Repo already present — pulling latest ...")
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--rebase"], check=True)

REPO_ROOT = CLONE_DIR
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"TASK            : {TASK}")
print(f"K_FOLDS         : {K_FOLDS}")
print(f"ENSEMBLE_METHOD : {ENSEMBLE_METHOD}")
print(f"N_QUBITS        : {N_QUBITS}")
print(f"Models ON       : {[k for k,v in RUN.items() if v]}")


## Section 2 — Install Dependencies

In [ ]:
import subprocess, sys

PKGS = [
    "pennylane", "pennylane-qiskit",
    "qiskit", "qiskit-machine-learning",
    "torchvision", "timm",
    "pandas", "openpyxl",
    "scikit-learn", "matplotlib",
    "tqdm", "einops", "scipy",
]
for pkg in PKGS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"  {'OK  ' if r.returncode==0 else 'WARN'} {pkg}")
print("Dependencies done.")


## Section 3 — Imports

In [ ]:
import os, sys, math, time, warnings, traceback, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.model_selection import (train_test_split,
    KFold, StratifiedKFold)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score,
    mean_absolute_error, f1_score, mean_squared_error)
from tqdm import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

torch.manual_seed(SEED)
np.random.seed(SEED)
# ── CUDA sanity check (guards against compute-capability mismatches) ──
DEVICE = "cpu"
if torch.cuda.is_available():
    try:
        _t = torch.zeros(1, device="cuda")
        _ = _t + _t          # trigger an actual kernel — catches arch mismatches
        del _t
        DEVICE = "cuda"
        print(f"  GPU : {torch.cuda.get_device_name(0)}")
    except Exception as _cuda_err:
        print(f"  ⚠ CUDA available but kernel mismatch → falling back to CPU")
        print(f"    ({_cuda_err})")
        torch.cuda.empty_cache()
print(f"Device  : {DEVICE}   PyTorch: {torch.__version__}")
print(f"Task    : {TASK}   K_FOLDS: {K_FOLDS}   Ensemble: {ENSEMBLE_METHOD}")


---
## Section 3b — Dataset, Test Split & Feature Extraction

**Important:** the test set is isolated first, before any K-fold split,
and is only used at the very end for final ensemble evaluation.


In [ ]:
_IMG_EXTS = [".jpg",".jpeg",".png",".bmp",".tif",".tiff",""]

def _find_img(pid, img_dir):
    for ext in _IMG_EXTS:
        p = os.path.join(img_dir, str(pid) + ext)
        if os.path.exists(p): return p
    return None

TRANSFORM_TRAIN = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
TRANSFORM_EVAL = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class HbDataset(Dataset):
    def __init__(self, records, img_dir, transform):
        self.records = records; self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        pid, hb = self.records[idx]
        path = _find_img(pid, self.img_dir)
        img  = Image.open(path).convert("RGB") if path else \
               Image.fromarray(np.zeros((224,224,3), dtype=np.uint8))
        img  = self.transform(img)
        if TASK == "classification":
            label = torch.tensor(1.0 if hb >= HB_THRESH else 0.0)
        else:
            lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
            label  = torch.tensor((hb-lo)/(hi-lo), dtype=torch.float32)
        return img, label

# ── Load & filter ─────────────────────────────────────────────
print("Loading dataset...")
df = pd.read_excel(EXCEL_PATH)
df = df[[IMAGE_COL, HB_COL]].dropna()
df[HB_COL] = pd.to_numeric(df[HB_COL], errors="coerce")  # force numeric; bad strings → NaN
df = df.dropna(subset=[HB_COL])                           # drop rows that failed conversion
df = df[(df[HB_COL]>=HB_FILTER_MIN)&(df[HB_COL]<=HB_FILTER_MAX)].reset_index(drop=True)
records = list(zip(df[IMAGE_COL].astype(str), df[HB_COL].astype(float)))
hb_vals = df[HB_COL].astype(float).values
print(f"  Total samples : {len(records)}")
print(f"  Hb range      : {hb_vals.min():.1f} – {hb_vals.max():.1f} g/dL")

# ── Hold-out test set (fixed, isolated first) ─────────────────
all_idx = np.arange(len(records))
if TASK == "classification":
    labels_strat = (hb_vals >= HB_THRESH).astype(int)
    tv_idx, te_idx = train_test_split(all_idx, test_size=TEST_SPLIT,
                                       random_state=SEED,
                                       stratify=labels_strat)
else:
    tv_idx, te_idx = train_test_split(all_idx, test_size=TEST_SPLIT,
                                       random_state=SEED)

tv_records = [records[i] for i in tv_idx]   # train+val (used in K-fold)
te_records = [records[i] for i in te_idx]   # held-out test set
tv_hb      = hb_vals[tv_idx]
te_hb      = hb_vals[te_idx]

print(f"  Train+Val : {len(tv_records)}  |  Test (held-out) : {len(te_records)}")

# Full datasets (for loader construction inside each fold)
full_dataset = HbDataset(records, IMAGE_DIR, TRANSFORM_EVAL)
test_dataset = HbDataset(te_records, IMAGE_DIR, TRANSFORM_EVAL)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)

print("Dataset & test split ready. ✅")


## Section 3c — ResNet-18 Backbone & Feature Caching

In [ ]:
# ── Shared ResNet-18 backbone ─────────────────────────────────
_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
_resnet.fc = nn.Identity()
_resnet = _resnet.to(DEVICE)
if FREEZE_BACKBONE:
    for p in _resnet.parameters(): p.requires_grad_(False)
_resnet.eval()
print(f"ResNet-18 ready  (frozen={FREEZE_BACKBONE})")

# ── Extract 512-dim features for the entire dataset once ──────
print("Pre-extracting ResNet features for all samples (once, shared)...")
_all_ds   = HbDataset(records, IMAGE_DIR, TRANSFORM_EVAL)
_all_ld   = DataLoader(_all_ds, batch_size=32, shuffle=False, num_workers=2)
ALL_FEATS = []   # (N, 512) numpy array
ALL_LABELS= []   # (N,)     numpy array — normalised hb or binary class

with torch.no_grad():
    for imgs, labels in tqdm(_all_ld, desc="ResNet", leave=False):
        ALL_FEATS.append(_resnet(imgs.to(DEVICE)).cpu().numpy())
        ALL_LABELS.append(labels.numpy())

ALL_FEATS  = np.concatenate(ALL_FEATS,  axis=0)   # (N, 512)
ALL_LABELS = np.concatenate(ALL_LABELS, axis=0)   # (N,)

# Split into train+val and test using the same indices
TV_FEATS   = ALL_FEATS[tv_idx]    # (Ntv, 512)
TV_LABELS  = ALL_LABELS[tv_idx]
TE_FEATS   = ALL_FEATS[te_idx]    # (Nte, 512)
TE_LABELS  = ALL_LABELS[te_idx]

# PCA → N_QUBITS (shared pipeline fitted on full train+val)
_sc  = StandardScaler()
_pca = PCA(n_components=N_QUBITS, random_state=SEED)
TV_FEATS_PCA = _pca.fit_transform(_sc.fit_transform(TV_FEATS))
TE_FEATS_PCA = _pca.transform(_sc.transform(TE_FEATS))
print(f"PCA variance retained: {_pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"TV_FEATS_PCA: {TV_FEATS_PCA.shape}  TE_FEATS_PCA: {TE_FEATS_PCA.shape}")
print("Feature extraction complete. ✅")


---
## Section 4 — K-Fold Engine, Ensemble Utilities & Metrics


In [ ]:
# ── Global OOF & test prediction stores ──────────────────────
#   OOF_PREDS  : dict  model_name → np.array (N_tv,)   OOF predictions
#   TEST_PREDS : dict  model_name → np.array (N_te,)   test predictions
#   CV_SCORES  : dict  model_name → float              mean CV metric
OOF_PREDS  = {}
TEST_PREDS = {}
CV_SCORES  = {}
MODEL_RESULTS = []    # per-fold detailed records

# ── Metrics ───────────────────────────────────────────────────
def _denorm_hb(v):
    lo,hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
    return np.array(v)*(hi-lo)+lo

def _reg_metrics(y_true_n, y_pred_n):
    yt = _denorm_hb(y_true_n); yp = _denorm_hb(y_pred_n)
    mae  = mean_absolute_error(yt, yp)
    rmse = mean_squared_error(yt, yp)**0.5
    return mae, rmse

def _cls_metrics(y_true, y_prob):
    yp = (np.array(y_prob)>=CLS_CONFIG["threshold"]).astype(int)
    acc = accuracy_score(y_true, yp)
    f1  = f1_score(y_true, yp, zero_division=0)
    try:    auc = roc_auc_score(y_true, y_prob)
    except: auc = float("nan")
    return acc, f1, auc

def _score(y_true, y_pred):
    """Primary CV score: lower = better for reg (MAE), higher = better for cls (AUC)."""
    if TASK=="regression":
        mae,_ = _reg_metrics(y_true, y_pred); return mae
    else:
        _,_,auc = _cls_metrics(y_true, y_pred); return auc

# ── Loss functions ────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy(logits, targets, reduction="none")
        return (((1-torch.exp(-bce))**self.gamma)*bce).mean()

def _loss_fn():
    if TASK=="classification":
        return FocalLoss(CLS_CONFIG["focal_gamma"]) if CLS_CONFIG["use_focal_loss"] \
               else nn.BCELoss()
    fn = REG_CONFIG["loss_fn"]
    if fn=="huber": return nn.HuberLoss(delta=REG_CONFIG["huber_delta"])
    if fn=="mae":   return nn.L1Loss()
    return nn.MSELoss()

# ── QuantumNet wrapper ────────────────────────────────────────
class QuantumNet(nn.Module):
    """Frozen ResNet-18 + Linear reducer + quantum model."""
    def __init__(self, qmodel):
        super().__init__()
        self.backbone = _resnet
        self.reducer  = nn.Linear(512, N_QUBITS)
        self.bn       = nn.BatchNorm1d(N_QUBITS)
        self.qmodel   = qmodel
    def forward(self, imgs):
        with torch.set_grad_enabled(not FREEZE_BACKBONE):
            f = self.backbone(imgs)
        x = torch.tanh(self.bn(self.reducer(f))) * math.pi
        return self.qmodel(x)

# ── FeatureDataset (for gradient models — wraps pre-extracted features) ──
class FeatDataset(Dataset):
    """Dataset of pre-extracted (512-dim) features + labels."""
    def __init__(self, feats, labels):
        self.X = torch.tensor(feats,   dtype=torch.float32)
        self.y = torch.tensor(labels,  dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

# ── Thin gradient model (feature input, no ResNet) ────────────
class ThinQuantumNet(nn.Module):
    """Takes pre-extracted 512-dim features directly."""
    def __init__(self, qmodel):
        super().__init__()
        self.reducer = nn.Linear(512, N_QUBITS)
        self.bn      = nn.BatchNorm1d(N_QUBITS)
        self.qmodel  = qmodel
    def forward(self, x):
        x = torch.tanh(self.bn(self.reducer(x))) * math.pi
        return self.qmodel(x)

# ── One gradient fold ─────────────────────────────────────────
def _train_gradient_fold(net, tr_feats, tr_labels, vl_feats, vl_labels,
                          n_epochs, desc=""):
    """Train ThinQuantumNet for one fold. Returns val predictions."""
    net = net.to(DEVICE)
    tr_dl = DataLoader(FeatDataset(tr_feats, tr_labels),
                       batch_size=BATCH_SIZE, shuffle=True)
    vl_dl = DataLoader(FeatDataset(vl_feats, vl_labels),
                       batch_size=BATCH_SIZE, shuffle=False)
    opt   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, net.parameters()), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    loss_fn = _loss_fn()
    best_val_loss = float("inf"); patience = 0
    best_state = None

    for epoch in range(n_epochs):
        net.train()
        for xb, yb in tr_dl:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(net(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        sched.step()

        # val loss for early stopping
        net.eval(); vl_loss = 0.0
        with torch.no_grad():
            for xb,yb in vl_dl:
                xb,yb = xb.to(DEVICE), yb.to(DEVICE)
                vl_loss += loss_fn(net(xb), yb).item()
        vl_loss /= max(1, len(vl_dl))
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_state    = copy.deepcopy(net.state_dict())
            patience = 0
        else:
            patience += 1
        if EARLY_STOP > 0 and patience >= EARLY_STOP: break

    # Predict on val fold
    if best_state: net.load_state_dict(best_state)
    net.eval()
    preds = []
    with torch.no_grad():
        for xb,_ in vl_dl:
            preds.extend(net(xb.to(DEVICE)).cpu().numpy())
    return net, np.array(preds)

def _predict_gradient(net, feats):
    """Inference with ThinQuantumNet."""
    ds = FeatDataset(feats, np.zeros(len(feats)))
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    net.eval(); preds = []
    with torch.no_grad():
        for xb,_ in dl:
            preds.extend(net(xb.to(DEVICE)).cpu().numpy())
    return np.array(preds)

def _add(folder):
    p = os.path.join(REPO_ROOT, folder)
    if p not in sys.path: sys.path.insert(0, p)

def _load_model(folder, task=TASK):
    import importlib.util as _ilu
    spec = _ilu.spec_from_file_location(
        f"_{folder}", os.path.join(REPO_ROOT, folder, "eye_hb_model.py"))
    mod = _ilu.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod

# ── K-Fold splitter ───────────────────────────────────────────
def _get_kfold():
    if TASK=="classification":
        tv_cls = (tv_hb >= HB_THRESH).astype(int)
        return StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED), tv_cls
    return KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED), tv_hb

print("Utilities ready. ✅")


---
## Section 5 — Per-Model K-Fold Runners

Two runners are defined:
- `run_kfold_gradient(name, folder)` — for PennyLane / TorchQuantum models
- `run_kfold_svm(name, folder)` — for Qiskit / sklearn-compatible models

Each runner:
1. Splits `TV_FEATS_PCA` into K folds
2. Trains on K-1 folds, collects OOF predictions on held-out fold
3. Reports per-fold metric + mean ± std
4. Retrains on the **full** Train+Val set (all K folds combined)
5. Predicts on the **held-out test set**


In [ ]:
def run_kfold_gradient(name, folder):
    """K-fold CV for gradient-based quantum models."""
    print(f"\n{'='*62}")
    print(f"  [GRADIENT] {name}")
    print(f"{'='*62}")
    t0 = time.time()
    kfold, strat_labels = _get_kfold()

    oof_preds  = np.zeros(len(TV_FEATS_PCA))
    fold_scores = []

    try:
        _add(folder)
        for fold, (tr_idx, vl_idx) in enumerate(
                kfold.split(TV_FEATS_PCA, strat_labels), 1):

            print(f"  ── Fold {fold}/{K_FOLDS}  "
                  f"(train={len(tr_idx)}, val={len(vl_idx)}) ──")

            tr_f, tr_l = TV_FEATS[tr_idx], TV_LABELS[tr_idx]  # 512-dim raw
            vl_f, vl_l = TV_FEATS[vl_idx], TV_LABELS[vl_idx]

            # Build fresh model for this fold
            mod = _load_model(folder)
            qm  = mod.build_model(n_features=N_QUBITS, n_qubits=N_QUBITS,
                                   n_layers=N_LAYERS, task=TASK)
            net = ThinQuantumNet(qm)

            # Use PCA features (N_QUBITS dim) for quantum input
            tr_pca = TV_FEATS_PCA[tr_idx]
            vl_pca = TV_FEATS_PCA[vl_idx]

            # Override ThinQuantumNet.reducer with identity (PCA already reduced)
            net.reducer = nn.Linear(N_QUBITS, N_QUBITS)
            nn.init.eye_(net.reducer.weight)
            nn.init.zeros_(net.reducer.bias)

            _, fold_preds = _train_gradient_fold(
                net, tr_pca, tr_l, vl_pca, vl_l, EPOCHS_PER_FOLD,
                desc=f"{name} fold {fold}")

            oof_preds[vl_idx] = fold_preds
            sc = _score(vl_l, fold_preds)
            fold_scores.append(sc)
            metric_name = "MAE(g/dL)" if TASK=="regression" else "AUC"
            fold_val = _reg_metrics(vl_l,fold_preds)[0] if TASK=="regression" \
                       else _cls_metrics(vl_l.astype(int), fold_preds)[2]
            print(f"     Fold {fold} {metric_name} = {fold_val:.4f}")

        cv_mean = np.mean(fold_scores)
        cv_std  = np.std(fold_scores)
        print(f"  CV {metric_name}: {cv_mean:.4f} ± {cv_std:.4f}")

        # ── Final retrain on full Train+Val ──────────────────
        print(f"  Retraining on full Train+Val ({len(TV_FEATS_PCA)} samples)...")
        mod  = _load_model(folder)
        qm   = mod.build_model(n_features=N_QUBITS, n_qubits=N_QUBITS,
                                n_layers=N_LAYERS, task=TASK)
        net  = ThinQuantumNet(qm)
        net.reducer = nn.Linear(N_QUBITS, N_QUBITS)
        nn.init.eye_(net.reducer.weight); nn.init.zeros_(net.reducer.bias)
        net, _ = _train_gradient_fold(
            net, TV_FEATS_PCA, TV_LABELS,
            TV_FEATS_PCA[:max(4,BATCH_SIZE)], TV_LABELS[:max(4,BATCH_SIZE)],
            EPOCHS_FINAL)

        # ── Predict on test set ──────────────────────────────
        test_preds = _predict_gradient(net, TE_FEATS_PCA)

        OOF_PREDS[name]  = oof_preds
        TEST_PREDS[name] = test_preds
        CV_SCORES[name]  = cv_mean

        elapsed = round(time.time()-t0, 1)
        if TASK=="regression":
            te_mae,te_rmse = _reg_metrics(TE_LABELS, test_preds)
            print(f"  ✅ Test  MAE={te_mae:.3f} g/dL  RMSE={te_rmse:.3f}  "
                  f"CV={cv_mean:.4f}±{cv_std:.4f}  ({elapsed}s)")
            MODEL_RESULTS.append({"name":name,"type":"gradient","status":"ok",
                "cv_score":round(cv_mean,4),"cv_std":round(cv_std,4),
                "test_mae":round(te_mae,3),"test_rmse":round(te_rmse,3),
                "time_s":elapsed})
        else:
            te_acc,te_f1,te_auc = _cls_metrics(TE_LABELS.astype(int), test_preds)
            print(f"  ✅ Test  acc={te_acc:.3f}  f1={te_f1:.3f}  auc={te_auc:.3f}  "
                  f"CV={cv_mean:.4f}±{cv_std:.4f}  ({elapsed}s)")
            MODEL_RESULTS.append({"name":name,"type":"gradient","status":"ok",
                "cv_score":round(cv_mean,4),"cv_std":round(cv_std,4),
                "test_acc":round(te_acc,3),"test_f1":round(te_f1,3),
                "test_auc":round(te_auc,3),"time_s":elapsed})

    except Exception as e:
        print(f"  ❌ FAILED: {e}"); traceback.print_exc()
        OOF_PREDS[name]  = np.full(len(TV_LABELS), np.nan)
        TEST_PREDS[name] = np.full(len(TE_LABELS), np.nan)
        CV_SCORES[name]  = np.nan
        MODEL_RESULTS.append({"name":name,"type":"gradient",
                               "status":"failed","error":str(e)[:100]})


def run_kfold_svm(name, folder):
    """K-fold CV for SVM/kernel-based quantum models (fit/predict interface)."""
    print(f"\n{'='*62}")
    print(f"  [SVM] {name}")
    print(f"{'='*62}")
    t0 = time.time()
    kfold, strat_labels = _get_kfold()

    oof_preds   = np.zeros(len(TV_FEATS_PCA))
    fold_scores = []

    # For classification convert normalised labels to binary
    tv_labels_svm = (TV_LABELS>=0.5).astype(int) if TASK=="classification" \
                    else TV_LABELS
    te_labels_svm = (TE_LABELS>=0.5).astype(int) if TASK=="classification" \
                    else TE_LABELS

    try:
        _add(folder)
        for fold, (tr_idx, vl_idx) in enumerate(
                kfold.split(TV_FEATS_PCA, strat_labels), 1):

            print(f"  ── Fold {fold}/{K_FOLDS}  "
                  f"(train={len(tr_idx)}, val={len(vl_idx)}) ──")

            mod = _load_model(folder)
            _, est = mod.build_model(n_qubits=N_QUBITS, task=TASK)

            est.fit(TV_FEATS_PCA[tr_idx], tv_labels_svm[tr_idx])

            if TASK=="classification" and hasattr(est,"predict_proba"):
                p = est.predict_proba(TV_FEATS_PCA[vl_idx])
                fold_preds = p[:,1] if p.ndim==2 else p
            else:
                fold_preds = est.predict(TV_FEATS_PCA[vl_idx]).astype(float)

            oof_preds[vl_idx] = fold_preds
            sc  = _score(tv_labels_svm[vl_idx], fold_preds)
            fold_scores.append(sc)
            metric_name = "MAE(g/dL)" if TASK=="regression" else "AUC"
            fv = _reg_metrics(TV_LABELS[vl_idx],fold_preds)[0] if TASK=="regression" \
                 else _cls_metrics(tv_labels_svm[vl_idx],fold_preds)[2]
            print(f"     Fold {fold} {metric_name} = {fv:.4f}")

        cv_mean = np.mean(fold_scores); cv_std = np.std(fold_scores)
        print(f"  CV {metric_name}: {cv_mean:.4f} ± {cv_std:.4f}")

        # ── Final retrain on full Train+Val ──────────────────
        print(f"  Retraining on full Train+Val...")
        mod  = _load_model(folder)
        _, est = mod.build_model(n_qubits=N_QUBITS, task=TASK)
        est.fit(TV_FEATS_PCA, tv_labels_svm)

        if TASK=="classification" and hasattr(est,"predict_proba"):
            tp = est.predict_proba(TE_FEATS_PCA)
            test_preds = tp[:,1] if tp.ndim==2 else tp
        else:
            test_preds = est.predict(TE_FEATS_PCA).astype(float)

        OOF_PREDS[name]  = oof_preds
        TEST_PREDS[name] = test_preds
        CV_SCORES[name]  = cv_mean

        elapsed = round(time.time()-t0, 1)
        if TASK=="regression":
            te_mae,te_rmse = _reg_metrics(TE_LABELS, test_preds)
            print(f"  ✅ Test  MAE={te_mae:.3f} g/dL  RMSE={te_rmse:.3f}  "
                  f"CV={cv_mean:.4f}±{cv_std:.4f}  ({elapsed}s)")
            MODEL_RESULTS.append({"name":name,"type":"svm","status":"ok",
                "cv_score":round(cv_mean,4),"cv_std":round(cv_std,4),
                "test_mae":round(te_mae,3),"test_rmse":round(te_rmse,3),
                "time_s":elapsed})
        else:
            te_acc,te_f1,te_auc = _cls_metrics(te_labels_svm, test_preds)
            print(f"  ✅ Test  acc={te_acc:.3f}  f1={te_f1:.3f}  auc={te_auc:.3f}  "
                  f"CV={cv_mean:.4f}±{cv_std:.4f}  ({elapsed}s)")
            MODEL_RESULTS.append({"name":name,"type":"svm","status":"ok",
                "cv_score":round(cv_mean,4),"cv_std":round(cv_std,4),
                "test_acc":round(te_acc,3),"test_f1":round(te_f1,3),
                "test_auc":round(te_auc,3),"time_s":elapsed})

    except Exception as e:
        print(f"  ❌ FAILED: {e}"); traceback.print_exc()
        OOF_PREDS[name]  = np.full(len(TV_LABELS), np.nan)
        TEST_PREDS[name] = np.full(len(TE_LABELS), np.nan)
        CV_SCORES[name]  = np.nan
        MODEL_RESULTS.append({"name":name,"type":"svm",
                               "status":"failed","error":str(e)[:100]})

print("K-fold runners ready. ✅")


---
## Section 6 — Run All Models (K-Fold)

### QSVM
*QSVM — Nature 567 (2019)*

In [ ]:
if RUN["QSVM"]:
    run_kfold_svm("QSVM", "01_QSVM_QuantumEnhancedFeatureSpaces")
else:
    print("QSVM skipped.")


### VQC
*VQC — arXiv 1802.06002 (2018)*

In [ ]:
if RUN["VQC"]:
    run_kfold_gradient("VQC", "02_VQC_VariationalQuantumClassifier")
else:
    print("VQC skipped.")


### QCNN
*QCNN — Nature Physics 15 (2019)*

In [ ]:
if RUN["QCNN"]:
    run_kfold_gradient("QCNN", "03_QCNN_QuantumConvolutionalNeuralNet")
else:
    print("QCNN skipped.")


### TorchQuantum_VQA
*TorchQuantum VQA — Nat. Rev. Physics (2021)*

In [ ]:
if RUN["TorchQuantum_VQA"]:
    run_kfold_gradient("TorchQuantum VQA", "04_VQA_TorchQuantum_Framework")
else:
    print("TorchQuantum VQA skipped.")


### DataReUploading
*Data Re-Uploading — Quantum 4,226 (2020)*

In [ ]:
if RUN["DataReUploading"]:
    run_kfold_gradient("Data Re-Uploading", "05_DataReUploading_UniversalClassifier")
else:
    print("Data Re-Uploading skipped.")


### QuantumTransferLearning
*Quantum Transfer Learning — Quantum 4,340 (2020)*

In [ ]:
if RUN["QuantumTransferLearning"]:
    run_kfold_gradient("Quantum Transfer Learning", "06_QuantumTransferLearning")
else:
    print("Quantum Transfer Learning skipped.")


### QuantumKernel
*Quantum Kernel — PRX Quantum (2021)*

In [ ]:
if RUN["QuantumKernel"]:
    run_kfold_svm("Quantum Kernel", "07_QuantumKernelMethods")
else:
    print("Quantum Kernel skipped.")


### QuantumRandForest
*Quantum Random Forest — Springer QMI (2023)*

In [ ]:
if RUN["QuantumRandForest"]:
    run_kfold_svm("Quantum Random Forest", "08_QuantumRandomForest")
else:
    print("Quantum Random Forest skipped.")


### QBoost
*QBoost — ACML 2012*

In [ ]:
if RUN["QBoost"]:
    run_kfold_svm("QBoost", "09_QBoost_QuantumBoosting")
else:
    print("QBoost skipped.")


### CVQNN
*CV-QNN — Phys Rev Research (2019)*

In [ ]:
if RUN["CVQNN"]:
    run_kfold_gradient("CV-QNN", "10_CVQNN_ContinuousVariableQNN")
else:
    print("CV-QNN skipped.")


### QSVM_Medical
*BONUS — QSVM Medical*

In [ ]:
if RUN["QSVM_Medical"]:
    run_kfold_svm("BONUS", "11_BONUS_QSVM_MedicalClassifier")
else:
    print("BONUS skipped.")


### QCNN_Phases
*BONUS — QCNN Phases (Cong et al.)*

In [ ]:
if RUN["QCNN_Phases"]:
    run_kfold_gradient("BONUS", "12_BONUS_QCNN_PhasesOfMatter")
else:
    print("BONUS skipped.")


### MetaLearning_QNN
*BONUS — Meta-Learning QNN*

In [ ]:
if RUN["MetaLearning_QNN"]:
    run_kfold_gradient("BONUS", "13_BONUS_VQA_MetaLearning_QNN")
else:
    print("BONUS skipped.")


---
## Section 7 — Build Ensemble & Final Test Evaluation

Three ensemble strategies applied to the collected OOF predictions:
1. **Simple Average** — equal weight to all models
2. **Weighted Average** — weight ∝ CV score (better model → higher weight)
3. **Stacking** — Ridge (regression) or LogisticRegression (classification)
   meta-learner trained on OOF predictions, predicts on test

The **held-out test set is only evaluated here** — it was never used during K-fold.


In [ ]:
import warnings, traceback
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
warnings.filterwarnings("ignore")

# ────────────────────────────────────────────────────────────────
#  BUILD ENSEMBLE INPUT MATRICES
# ────────────────────────────────────────────────────────────────
valid_names = [n for n, p in OOF_PREDS.items()
               if not np.isnan(p).any() and not np.isnan(CV_SCORES[n])]
skipped     = [n for n in OOF_PREDS if n not in valid_names]

print("=" * 70)
print(f"  ENSEMBLE ENGINE  —  Task: {TASK.upper()}  |  K={K_FOLDS} folds")
print("=" * 70)
print(f"  Valid models  : {len(valid_names)}  → {valid_names}")
if skipped:
    print(f"  Skipped/failed: {skipped}")
print("-" * 70)

if len(valid_names) == 0:
    print("  ✗  No valid models — cannot build ensemble.")
else:
    OOF_mat  = np.stack([OOF_PREDS[n]  for n in valid_names], axis=1)  # (Ntv, M)
    TEST_mat = np.stack([TEST_PREDS[n] for n in valid_names], axis=1)  # (Nte, M)
    tv_y = (TV_LABELS >= 0.5).astype(int) if TASK == "classification" else TV_LABELS
    te_y = (TE_LABELS >= 0.5).astype(int) if TASK == "classification" else TE_LABELS

    cv_scores_arr = np.array([CV_SCORES[n] for n in valid_names])

    ensemble_results = {}   # method_key → dict of metrics

    # ── Helper: compute and log metrics ──────────────────────────────────
    def _eval_ensemble(key, label, oof_pred, test_pred):
        oof_sc = _score(tv_y, oof_pred)
        if TASK == "regression":
            te_mae, te_rmse = _reg_metrics(TE_LABELS, test_pred)
            ensemble_results[key] = {
                "label": label, "oof_score": oof_sc,
                "test_mae": te_mae, "test_rmse": te_rmse,
            }
            print(f"  [{label:<24s}]  OOF MAE={oof_sc:.4f}  "
                  f"Test MAE={te_mae:.3f} g/dL  RMSE={te_rmse:.3f}")
        else:
            te_acc, te_f1, te_auc = _cls_metrics(te_y, test_pred)
            ensemble_results[key] = {
                "label": label, "oof_score": oof_sc,
                "test_acc": te_acc, "test_f1": te_f1, "test_auc": te_auc,
            }
            print(f"  [{label:<24s}]  OOF AUC={oof_sc:.4f}  "
                  f"Acc={te_acc:.3f}  F1={te_f1:.3f}  AUC={te_auc:.3f}")

    # ── 1. Simple Average ────────────────────────────────────────────────
    if RUN_ENSEMBLE.get("simple_avg", False):
        _eval_ensemble("simple_avg", "Simple Average",
                       OOF_mat.mean(axis=1), TEST_mat.mean(axis=1))

    # ── 2. Weighted Average ──────────────────────────────────────────────
    if RUN_ENSEMBLE.get("weighted_avg", False):
        w = (1.0 / (cv_scores_arr + 1e-9)) if TASK == "regression" else cv_scores_arr
        w = w / w.sum()
        _eval_ensemble("weighted_avg", "Weighted Average",
                       (OOF_mat * w).sum(axis=1), (TEST_mat * w).sum(axis=1))

    # ── 3. Rank Average ──────────────────────────────────────────────────
    if RUN_ENSEMBLE.get("rank_avg", False):
        from scipy.stats import rankdata
        oof_ranks  = np.stack([rankdata(OOF_mat[:, i])  for i in range(OOF_mat.shape[1])],  axis=1)
        test_ranks = np.stack([rankdata(TEST_mat[:, i]) for i in range(TEST_mat.shape[1])], axis=1)
        # map mean rank back to [0,1] range
        oof_rank_avg  = (oof_ranks.mean(axis=1)  - 1) / max(oof_ranks.shape[0]  - 1, 1)
        test_rank_avg = (test_ranks.mean(axis=1) - 1) / max(test_ranks.shape[0] - 1, 1)
        if TASK == "regression":
            # convert rank-fraction back to Hb scale for metric computation
            lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
            oof_ra_scaled  = oof_rank_avg  * (hi - lo) + lo
            test_ra_scaled = test_rank_avg * (hi - lo) + lo
            # for _score we need normalised, so keep as-is
            _eval_ensemble("rank_avg", "Rank Average", oof_rank_avg, test_rank_avg)
        else:
            _eval_ensemble("rank_avg", "Rank Average", oof_rank_avg, test_rank_avg)

    # ── 4. Stacking — Ridge / LogReg ────────────────────────────────────
    if RUN_ENSEMBLE.get("stacking_ridge", False):
        try:
            if TASK == "regression":
                meta = Ridge(alpha=1.0)
                meta.fit(OOF_mat, tv_y)
                _eval_ensemble("stacking_ridge", "Stacking (Ridge)",
                               meta.predict(OOF_mat), meta.predict(TEST_mat))
            else:
                meta = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
                meta.fit(OOF_mat, tv_y)
                _eval_ensemble("stacking_ridge", "Stacking (LogReg)",
                               meta.predict_proba(OOF_mat)[:, 1],
                               meta.predict_proba(TEST_mat)[:, 1])
        except Exception as e:
            print(f"  [Stacking Ridge/LogReg] FAILED — {e}")

    # ── 5. Stacking — Gradient Boosting ─────────────────────────────────
    if RUN_ENSEMBLE.get("stacking_gbm", False):
        try:
            if TASK == "regression":
                meta = GradientBoostingRegressor(
                    n_estimators=100, max_depth=3, random_state=SEED)
                meta.fit(OOF_mat, tv_y)
                _eval_ensemble("stacking_gbm", "Stacking (GBM)",
                               meta.predict(OOF_mat), meta.predict(TEST_mat))
            else:
                meta = GradientBoostingClassifier(
                    n_estimators=100, max_depth=3, random_state=SEED)
                meta.fit(OOF_mat, tv_y)
                _eval_ensemble("stacking_gbm", "Stacking (GBM)",
                               meta.predict_proba(OOF_mat)[:, 1],
                               meta.predict_proba(TEST_mat)[:, 1])
        except Exception as e:
            print(f"  [Stacking GBM] FAILED — {e}")

    # ── 6. Voting / Median ───────────────────────────────────────────────
    if RUN_ENSEMBLE.get("voting", False):
        if TASK == "regression":
            # Median is more robust than mean to outlier models
            _eval_ensemble("voting", "Median Vote (Reg)",
                           np.median(OOF_mat, axis=1), np.median(TEST_mat, axis=1))
        else:
            # Hard majority vote for classification
            thresh = CLS_CONFIG["threshold"]
            oof_votes  = (OOF_mat  >= thresh).astype(int).mean(axis=1)
            test_votes = (TEST_mat >= thresh).astype(int).mean(axis=1)
            _eval_ensemble("voting", "Majority Vote (Cls)", oof_votes, test_votes)

    # ── 7. Best Single (reference) ───────────────────────────────────────
    if RUN_ENSEMBLE.get("best_single", False):
        if TASK == "regression":
            best_idx = int(np.argmin(cv_scores_arr))
        else:
            best_idx = int(np.argmax(cv_scores_arr))
        best_name = valid_names[best_idx]
        _eval_ensemble("best_single", f"Best Single ({best_name[:10]})",
                       OOF_mat[:, best_idx], TEST_mat[:, best_idx])

    print("-" * 70)
    print(f"  Ensembles completed: {len(ensemble_results)}")


---
## Section 8 — Full Results Summary & Plots

In [ ]:
# ════════════════════════════════════════════════════════════════
#   RESULTS DASHBOARD
# ════════════════════════════════════════════════════════════════
import textwrap, math

SEP  = "═" * 72
sep  = "─" * 72
W    = 72

def _banner(title):
    pad = (W - len(title) - 2) // 2
    print(f"\n{SEP}")
    print(f"{'═'*pad} {title} {'═'*(W - pad - len(title) - 2)}")
    print(SEP)

def _medal(rank):
    return ["🥇","🥈","🥉"][rank] if rank < 3 else f"  {rank+1}."

# ────────────────────────────────────────────────────────────────
#  A.  PER-MODEL K-FOLD RESULTS
# ────────────────────────────────────────────────────────────────
if MODEL_RESULTS:
    _banner(f"PER-MODEL K={K_FOLDS}-FOLD RESULTS  ·  Task: {TASK.upper()}")
    df_m = pd.DataFrame(MODEL_RESULTS)

    if TASK == "regression":
        sort_col, asc = "test_mae", True
        metric_cols   = ["cv_score","cv_std","test_mae","test_rmse"]
        metric_hdr    = f"{'CV MAE':>9}{'±':>4}{'Test MAE':>10}{'RMSE':>8}"
    else:
        sort_col, asc = "test_auc", False
        metric_cols   = ["cv_score","cv_std","test_acc","test_f1","test_auc"]
        metric_hdr    = f"{'CV AUC':>9}{'±':>4}{'Acc':>8}{'F1':>8}{'AUC':>8}"

    ok_rows = df_m[df_m["status"]=="ok"].sort_values(sort_col, ascending=asc)
    bad_rows = df_m[df_m["status"]!="ok"]

    print(f"\n  {'Rank':<5} {'Model':<28} {'Type':<10} {metric_hdr}  {'Time':>6}")
    print(f"  {sep}")
    for rank, (_, row) in enumerate(ok_rows.iterrows()):
        cv  = f"{row['cv_score']:.4f}"
        std = f"{row.get('cv_std',0):.4f}"
        if TASK == "regression":
            metrics = (f"{row.get('test_mae',float('nan')):>10.3f}"
                       f"{row.get('test_rmse',float('nan')):>8.3f}")
        else:
            metrics = (f"{row.get('test_acc',float('nan')):>8.3f}"
                       f"{row.get('test_f1',float('nan')):>8.3f}"
                       f"{row.get('test_auc',float('nan')):>8.3f}")
        t = row.get("time_s", 0)
        time_str = f"{t/60:.1f}m" if t >= 60 else f"{t:.1f}s"
        print(f"  {_medal(rank):<5} {row['name']:<28} {row.get('type','?'):<10} "
              f"{cv:>9}{std:>5}{metrics}  {time_str:>6}")

    if not bad_rows.empty:
        print(f"\n  Failed / skipped:")
        for _, row in bad_rows.iterrows():
            print(f"    ✗  {row['name']:<28}  {row.get('error_msg','—')[:40]}")

# ────────────────────────────────────────────────────────────────
#  B.  ENSEMBLE LEADERBOARD
# ────────────────────────────────────────────────────────────────
if "ensemble_results" in dir() and ensemble_results:
    _banner("ENSEMBLE LEADERBOARD  ·  Ranked by Test Performance")

    rows = list(ensemble_results.values())
    if TASK == "regression":
        rows.sort(key=lambda r: r.get("test_mae", 9999))
        print(f"\n  {'Rank':<5} {'Ensemble Method':<28} {'OOF MAE':>9} {'Test MAE':>10} {'RMSE':>8}")
        print(f"  {sep}")
        for rank, r in enumerate(rows):
            flag = "  ← BEST" if rank == 0 else ""
            print(f"  {_medal(rank):<5} {r['label']:<28}"
                  f"{r['oof_score']:>9.4f}"
                  f"{r.get('test_mae',float('nan')):>10.3f}"
                  f"{r.get('test_rmse',float('nan')):>8.3f}{flag}")
    else:
        rows.sort(key=lambda r: r.get("test_auc", 0), reverse=True)
        print(f"\n  {'Rank':<5} {'Ensemble Method':<28} {'OOF AUC':>9} {'Acc':>8} {'F1':>8} {'AUC':>8}")
        print(f"  {sep}")
        for rank, r in enumerate(rows):
            flag = "  ← BEST" if rank == 0 else ""
            print(f"  {_medal(rank):<5} {r['label']:<28}"
                  f"{r['oof_score']:>9.4f}"
                  f"{r.get('test_acc',float('nan')):>8.3f}"
                  f"{r.get('test_f1',float('nan')):>8.3f}"
                  f"{r.get('test_auc',float('nan')):>8.3f}{flag}")

    # pick winner
    if TASK == "regression":
        winner = min(rows, key=lambda r: r.get("test_mae", 9999))
        wval   = f"MAE = {winner.get('test_mae', float('nan')):.3f} g/dL"
    else:
        winner = max(rows, key=lambda r: r.get("test_auc", 0))
        wval   = f"AUC = {winner.get('test_auc', float('nan')):.3f}"
    print(f"\n  🏆  Best ensemble → [{winner['label']}]   {wval}")

# ────────────────────────────────────────────────────────────────
#  C.  ENSEMBLE vs BEST SINGLE MODEL  (side-by-side)
# ────────────────────────────────────────────────────────────────
if "ensemble_results" in dir() and ensemble_results and MODEL_RESULTS:
    _banner("ENSEMBLE  vs  BEST SINGLE MODEL")
    df_ok2 = pd.DataFrame([r for r in MODEL_RESULTS if r.get("status") == "ok"])
    if not df_ok2.empty:
        if TASK == "regression":
            best_m = df_ok2.loc[df_ok2["test_mae"].idxmin()]
            best_e = min(ensemble_results.values(), key=lambda r: r.get("test_mae", 9999))
            m_val, e_val = best_m["test_mae"], best_e.get("test_mae", float("nan"))
            gain = ((m_val - e_val) / m_val * 100) if m_val != 0 else 0
            print(f"\n  Best single model : {best_m['name']:<28}  Test MAE = {m_val:.3f} g/dL")
            print(f"  Best ensemble     : {best_e['label']:<28}  Test MAE = {e_val:.3f} g/dL")
            if gain > 0:
                print(f"  → Ensemble improves MAE by {gain:.1f}%  🎉")
            else:
                print(f"  → Single model is better by {-gain:.1f}%  (consider more diverse models)")
        else:
            best_m = df_ok2.loc[df_ok2["test_auc"].idxmax()]
            best_e = max(ensemble_results.values(), key=lambda r: r.get("test_auc", 0))
            m_val, e_val = best_m["test_auc"], best_e.get("test_auc", float("nan"))
            gain = ((e_val - m_val) / max(m_val, 1e-9) * 100)
            print(f"\n  Best single model : {best_m['name']:<28}  Test AUC = {m_val:.3f}")
            print(f"  Best ensemble     : {best_e['label']:<28}  Test AUC = {e_val:.3f}")
            if gain > 0:
                print(f"  → Ensemble improves AUC by {gain:.1f}%  🎉")
            else:
                print(f"  → Single model is better by {-gain:.1f}%  (consider more diverse models)")

# ────────────────────────────────────────────────────────────────
#  D.  MODEL DIVERSITY SUMMARY
# ────────────────────────────────────────────────────────────────
valid_oof = [n for n, p in OOF_PREDS.items() if not np.isnan(p).any()]
if len(valid_oof) > 1:
    _banner("MODEL DIVERSITY  (OOF Prediction Correlation)")
    oof_df   = pd.DataFrame({n: OOF_PREDS[n] for n in valid_oof})
    corr_mat = oof_df.corr()
    upper    = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
    vals     = upper.stack().values
    print(f"\n  Mean pairwise Pearson r : {vals.mean():.3f}")
    print(f"  Min / Max correlation   : {vals.min():.3f} / {vals.max():.3f}")
    print(f"  (Ideally mean r < 0.85 for diverse ensemble)")

    # Most & least correlated pairs
    pairs = [(corr_mat.iloc[i,j], valid_oof[i], valid_oof[j])
             for i in range(len(valid_oof))
             for j in range(i+1, len(valid_oof))]
    pairs.sort(key=lambda x: x[0])
    print(f"\n  Most diverse pair  : {pairs[0][1]:<28} ↔ {pairs[0][2]:<28}  r={pairs[0][0]:.3f}")
    print(f"  Most similar pair  : {pairs[-1][1]:<28} ↔ {pairs[-1][2]:<28}  r={pairs[-1][0]:.3f}")

# ────────────────────────────────────────────────────────────────
#  E.  VISUALIZATION — 3-panel figure
# ────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

ok_model_rows = [r for r in MODEL_RESULTS if r.get("status") == "ok"]
has_ensemble  = "ensemble_results" in dir() and ensemble_results

if ok_model_rows or has_ensemble:
    fig = plt.figure(figsize=(20, 12))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
    ax1 = fig.add_subplot(gs[0, :2])   # top-left wide: model bar chart
    ax2 = fig.add_subplot(gs[0, 2])    # top-right: CV vs test scatter
    ax3 = fig.add_subplot(gs[1, :2])   # bottom-left wide: ensemble bar chart
    ax4 = fig.add_subplot(gs[1, 2])    # bottom-right: OOF correlation heatmap

    metric = "test_mae" if TASK == "regression" else "test_auc"
    xlabel = "Test MAE (g/dL)" if TASK == "regression" else "Test AUC"

    # ── Panel 1: Per-model test metric ──────────────────────────────
    if ok_model_rows:
        df_ok = pd.DataFrame(ok_model_rows).sort_values(metric, ascending=(TASK=="regression"))
        colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(df_ok)))
        bars = ax1.barh(df_ok["name"], df_ok[metric], color=colors, edgecolor="white", linewidth=0.5)
        for bar, val in zip(bars, df_ok[metric]):
            ax1.text(val + (0.002 if TASK=="regression" else 0.001), bar.get_y() + bar.get_height()/2,
                     f"{val:.3f}", va="center", ha="left", fontsize=7.5, fontweight="bold")
        ax1.set_xlabel(xlabel, fontsize=10)
        ax1.set_title(f"Per-Model Test {xlabel}  (sorted best → worst)", fontsize=10, fontweight="bold")
        ax1.grid(axis="x", alpha=0.3, linestyle="--")
        ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

    # ── Panel 2: CV vs Test scatter ─────────────────────────────────
    if ok_model_rows:
        ax2.scatter(df_ok["cv_score"], df_ok[metric],
                    s=90, c=plt.cm.plasma(np.linspace(0.15,0.85,len(df_ok))),
                    edgecolors="k", linewidths=0.4, zorder=3)
        for _, row in df_ok.iterrows():
            ax2.annotate(row["name"][:10], (row["cv_score"], row[metric]),
                         fontsize=6.5, xytext=(4, 2), textcoords="offset points")
        ax2.set_xlabel("CV Score", fontsize=9); ax2.set_ylabel(xlabel, fontsize=9)
        ax2.set_title("CV Score vs Test Performance", fontsize=10, fontweight="bold")
        ax2.grid(True, alpha=0.3, linestyle="--")
        ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

    # ── Panel 3: Ensemble leaderboard bars ──────────────────────────
    if has_ensemble:
        ens_rows = sorted(ensemble_results.values(),
                          key=lambda r: r.get(metric, 9999 if TASK=="regression" else 0),
                          reverse=(TASK != "regression"))
        ens_labels = [r["label"] for r in ens_rows]
        ens_vals   = [r.get(metric, float("nan")) for r in ens_rows]
        ens_colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(ens_rows)))
        bars2 = ax3.barh(ens_labels, ens_vals, color=ens_colors, edgecolor="white", linewidth=0.5)
        for bar, val in zip(bars2, ens_vals):
            if not math.isnan(val):
                ax3.text(val + (0.002 if TASK=="regression" else 0.001),
                         bar.get_y() + bar.get_height()/2,
                         f"{val:.3f}", va="center", ha="left", fontsize=7.5, fontweight="bold")
        ax3.set_xlabel(xlabel, fontsize=10)
        ax3.set_title(f"Ensemble Methods  —  Test {xlabel}  (sorted best → worst)", fontsize=10, fontweight="bold")
        ax3.grid(axis="x", alpha=0.3, linestyle="--")
        ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
        # mark best with gold edge
        bars2[0].set_edgecolor("gold"); bars2[0].set_linewidth(2.5)

    # ── Panel 4: OOF correlation heatmap ────────────────────────────
    if len(valid_oof) > 1:
        corr = pd.DataFrame({n: OOF_PREDS[n] for n in valid_oof}).corr()
        n    = len(valid_oof)
        im   = ax4.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
        ax4.set_xticks(range(n)); ax4.set_xticklabels(
            [x[:8] for x in valid_oof], rotation=45, ha="right", fontsize=6)
        ax4.set_yticks(range(n)); ax4.set_yticklabels(
            [x[:8] for x in valid_oof], fontsize=6)
        for i in range(n):
            for j in range(n):
                ax4.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center",
                         fontsize=5.5, color="k" if abs(corr.iloc[i,j]) < 0.7 else "w")
        plt.colorbar(im, ax=ax4, label="r", shrink=0.85)
        ax4.set_title("OOF Correlation\n(low r = diverse ensemble)", fontsize=9, fontweight="bold")

    fig.suptitle(f"QuantumHb — Ensemble Dashboard  |  {TASK.upper()}  |  K={K_FOLDS} folds",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.savefig(os.path.join(REPO_ROOT, "ensemble_results.png"), dpi=130, bbox_inches="tight")
    plt.show()
    print("\n  📊  Dashboard saved → ensemble_results.png")

print(f"\n{SEP}")
print(f"  Done. Ran {len(ensemble_results) if 'ensemble_results' in dir() else 0} ensemble(s)")
print(SEP)


In [ ]:
# ── Training Curves ─────────────────────────────────────────────
import matplotlib.pyplot as plt

loss_history = [r.get('tr_loss', 0) for r in RESULTS] if 'RESULTS' in dir() else []
acc_history  = [r.get('test_acc', r.get('test_mae', 0)) for r in RESULTS] if 'RESULTS' in dir() else []

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
if 'loss_history' in dir() or 'loss_history' in locals() or 'loss_history' in globals():
    axes[0].plot(loss_history, color='royalblue', linewidth=2)
    axes[0].set_title('Training Loss', fontsize=13)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'Loss history\nnot available',
                 ha='center', va='center', transform=axes[0].transAxes)

# Accuracy
if 'acc_history' in dir() or 'acc_history' in locals() or 'acc_history' in globals():
    axes[1].plot(acc_history, color='darkorange', linewidth=2)
    axes[1].set_title('Training Accuracy', fontsize=13)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Accuracy history\nnot available',
                 ha='center', va='center', transform=axes[1].transAxes)

plt.suptitle('Training Summary', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Training curves saved to training_curves.png')
